EXERCICE TITANIC

In [77]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

In [78]:
titanic = sns.load_dataset('titanic')
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [79]:
titanic.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [80]:
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = titanic[features]
y = titanic['survived'].values

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [82]:
age_median = X_train['age'].median()
embarked_mode = X_train['embarked'].mode()[0]

X_train = X_train.fillna({'age': age_median, 'embarked': embarked_mode})
X_test = X_test.fillna({'age': age_median, 'embarked': embarked_mode})

In [83]:
#Encoder les variables catégorielles
cat = ['sex', 'embarked', 'pclass']
X_train = pd.get_dummies(X_train, columns=cat, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat, drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [84]:
# Normaliser les features numériques
num = ['age', 'fare']
scaler = StandardScaler()
X_train[num] = scaler.fit_transform(X_train[num])
X_test[num] = scaler.transform(X_test[num])


In [85]:
# Conversion en numpy array
X_train = X_train.values.astype('float32')
X_test = X_test.values.astype('float32')
print("Shape X_train:", X_train.shape, "| Shape X_test:", X_test.shape)

Shape X_train: (712, 9) | Shape X_test: (179, 9)


In [86]:

# Construire le modèle
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),  # couche cachée 27 neurones avec ReLU
    Dense(1, activation='sigmoid') # en sorti la sigmoid
])

In [87]:
# Compiler le modèle
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [88]:
early = EarlyStopping(patience=10, restore_best_weights=True)

In [89]:
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early],
    verbose=0
)

In [90]:
predictions = model.predict(X_test)

y_pred = (predictions > 0.5).astype(int).ravel()
print(classification_report(y_test, y_pred))
print(history.history)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       110
           1       0.79      0.64      0.70        69

    accuracy                           0.79       179
   macro avg       0.79      0.76      0.77       179
weighted avg       0.79      0.79      0.79       179

{'accuracy': [0.503125011920929, 0.675000011920929, 0.690625011920929, 0.7093750238418579, 0.71875, 0.739062488079071, 0.754687488079071, 0.7640625238418579, 0.7796875238418579, 0.7984374761581421, 0.809374988079071, 0.815625011920929, 0.8203125, 0.8296874761581421, 0.8265625238418579, 0.828125, 0.823437511920929, 0.828125, 0.832812488079071, 0.831250011920929, 0.8296874761581421, 0.828125, 0.832812488079071, 0.832812488079071, 0.8343750238418579, 0.832812488079071, 0.8374999761581421, 0.8374999761581421, 0.839062511920929, 0.839062511920929, 0.840624988079071, 0.840624988079071, 0.840624988079071, 0.8421875238418579, 0.8421875